# Clase 016 — NumPy: agregaciones

**Parte 0** · VanderPlas cap. 2 § 2.4.

> 🎯 Reducir arrays a estadísticos con el `axis` correcto — la fuente del 50% de los bugs de orientación.

> ⏱️ ~60 min

## ⚙️ Setup

In [ ]:
import numpy as np
rng = np.random.default_rng(42)

## 1️⃣ Reducciones básicas

```python
arr.sum()      arr.mean()     arr.std()     arr.var()
arr.min()      arr.max()      np.median(arr)
np.percentile(arr, 25)         np.percentile(arr, [25, 50, 75])
```

Método del array (`arr.sum()`) o función NumPy (`np.sum(arr)`) — equivalentes.

In [ ]:
x = rng.normal(0, 1, 1000)
print(f'mean   : {x.mean():.4f}')
print(f'std    : {x.std():.4f}')
print(f'median : {np.median(x):.4f}')
print(f'p25,75 : {np.percentile(x, [25, 75])}')
print(f'min,max: ({x.min():.2f}, {x.max():.2f})')

## 2️⃣ El bug del `axis`

Dada una matriz `(rows, cols)`:

- `axis=0` → reduce **filas**, deja **una valor por columna**.
- `axis=1` → reduce **columnas**, deja **un valor por fila**.

Mnemónico: "el axis que pasas es el que **desaparece**".

In [ ]:
# Matriz 4 filas × 3 columnas
M = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
    [10, 11, 12],
])
print(f'M.shape = {M.shape}')
print(f'M.sum()         → escalar : {M.sum()}')
print(f'M.sum(axis=0)   → por col : {M.sum(axis=0)}  shape {M.sum(axis=0).shape}')
print(f'M.sum(axis=1)   → por fila: {M.sum(axis=1)}  shape {M.sum(axis=1).shape}')

## 3️⃣ Caso típico: ventas día × tienda

Matriz 100 días × 4 tiendas:

In [ ]:
ventas = rng.integers(50, 500, size=(100, 4))
print(f'ventas.shape = {ventas.shape}')

print('\n— por tienda (resumen vertical, axis=0) —')
print(f'media : {ventas.mean(axis=0).round(1)}')
print(f'total : {ventas.sum(axis=0)}')
print(f'mejor : {ventas.max(axis=0)}')

print('\n— por día (resumen horizontal, axis=1) — primeros 5 días —')
print(f'media diaria primeros 5 días: {ventas.mean(axis=1)[:5].round(1)}')

## 4️⃣ Variantes NaN-aware

| Sin NaN | Con NaN |
|---|---|
| `sum`, `mean`, `std`, `var` | `nansum`, `nanmean`, `nanstd`, `nanvar` |
| `min`, `max`, `median` | `nanmin`, `nanmax`, `nanmedian` |
| `argmin`, `argmax` | `nanargmin`, `nanargmax` |
| `percentile` | `nanpercentile` |

In [ ]:
datos = rng.normal(0, 1, 100).copy()
idx_nans = rng.choice(100, 10, replace=False)
datos[idx_nans] = np.nan

print(f'mean (propaga) : {datos.mean()}')
print(f'nanmean        : {np.nanmean(datos):.4f}')
print(f'nanmedian      : {np.nanmedian(datos):.4f}')
print(f'NaN count      : {np.isnan(datos).sum()}')

## 5️⃣ Acumulativas: `cumsum`, `cumprod`

Útiles para series temporales — precio acumulado, drawdown, totales corridos:

In [ ]:
# Retornos diarios pequeños
retornos = rng.normal(0.001, 0.02, 30)
print('retornos:', retornos[:5].round(4), '...')

# Precio acumulado partiendo de 100
precio = 100 * np.cumprod(1 + retornos)
print(f'precio final: {precio[-1]:.2f}')
print(f'precio max  : {precio.max():.2f}')

# Suma acumulada
ventas_diarias = rng.integers(50, 200, 30)
print(f'\nventa total acumulada día 30: {ventas_diarias.cumsum()[-1]}')

## 6️⃣ `argmin` / `argmax` — posición del extremo

No el **valor**, el **índice**:

In [ ]:
x = rng.normal(0, 1, 10)
print('array      :', x.round(2))
print(f'max        : {x.max():.4f}')
print(f'argmax     : {x.argmax()}  ← índice del max')
print(f'verificación: x[{x.argmax()}] = {x[x.argmax()]:.4f}')

# En matriz: por eje
M = rng.normal(0, 1, (5, 3))
print(f'\nM.argmax(axis=0) → fila del max por columna: {M.argmax(axis=0)}')
print(f'M.argmax(axis=1) → col del max por fila    : {M.argmax(axis=1)}')

## 7️⃣ `all` y `any` — reducciones booleanas

```python
(arr > 0).all()        # ¿todos > 0?
(arr > 0).any()        # ¿al menos uno > 0?
(M > 0).all(axis=1)    # ¿todas las cols positivas en cada fila?
```

## ✅ Checklist

- [ ] Conozco sum/mean/std/median/percentile
- [ ] Sé que `axis=0` reduce filas (resultado por columna)
- [ ] Uso variantes `nan*` cuando hay NaN
- [ ] Uso `cumsum`/`cumprod` para series
- [ ] Uso `argmax`/`argmin` para encontrar posiciones

## 📝 Homework

Ver `README.md`. Matriz 365×5 de ventas con análisis completo (media/std por tienda, mejor/peor día, cumsum anual, manejo NaN).

## 📖 Definiciones y características

**Agregación / reducción**

Operación que **colapsa** un array a menos dimensiones: `sum`, `mean`, `std`, `min`, `max`, `argmax`. Sin `axis`, reduce a un escalar; con `axis=N`, elimina la dimensión N.

**`axis=0` vs `axis=1`**

**Regla**: el axis que pasas es el que **desaparece**. En matriz `(filas, cols)`: `axis=0` colapsa filas → un valor por columna; `axis=1` colapsa cols → un valor por fila. Mnemónico inverso al que muchos esperan.

**`argmin` / `argmax`**

Devuelven el **índice** del extremo (no el valor). `arr.argmax()` = posición del máximo; `arr[arr.argmax()]` = valor máximo. Con `axis=` devuelven array de índices por fila/columna.

**Variantes `nan*`**

Versiones que **ignoran NaN** en vez de propagarlo: `nansum`, `nanmean`, `nanstd`, `nanmin`, `nanmax`, `nanmedian`, `nanpercentile`, `nanargmax`. Útiles cuando los datos tienen missing.

**Acumulativas (`cumsum`, `cumprod`)**

No colapsan — devuelven array de igual shape con valores acumulados hasta cada posición. Útiles para series temporales (precio acumulado, drawdown).

**`percentile` / `quantile`**

Valor por debajo del cual cae el N% de los datos. `np.percentile(arr, 50)` = mediana. `np.percentile(arr, [25, 50, 75])` = cuartiles.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `M.sum(axis=0)` da resultado por columna y esperaba por fila | Confusión clásica. **Regla**: `axis=0` reduce **filas** (resultado per-column). **Fix**: si querías por fila, usa `axis=1`. Memoriza: "el axis que pasas desaparece". |
| `arr.mean()` da NaN aunque solo hay un par de NaN | Cualquier NaN propaga. **Fix**: `np.nanmean(arr)` o filtra antes (`arr[~np.isnan(arr)]`). |
| `np.argmax(matriz)` devuelve un solo número raro | Sin `axis`, aplana el array primero y devuelve un índice **lineal**. **Fix**: `argmax(axis=0)` o `axis=1` para obtener índices por dimensión; o `np.unravel_index(argmax, shape)` para coordenadas. |
| `np.percentile([1,2,3,4], 50)` da `2.5` no `2` | Interpolación lineal por default. Si quieres el valor exacto del dataset, usa `interpolation='nearest'` o `quantile` con `method='lower'`. |
| `cumsum` de floats acumula error de redondeo en arrays largos | Sumas múltiples introducen error numérico. **Fix**: si necesitas precisión, `np.cumsum(arr, dtype=np.float64)` o usa pairwise summation (Kahan). |

## ❓ Preguntas frecuentes

**❓ ¿`arr.sum()` o `np.sum(arr)`?**

Equivalentes. Método del array (`.sum()`) es más legible en cadenas (`arr.clip(0).sum()`). Función (`np.sum`) acepta también listas (no solo ndarray).

**❓ ¿Cómo recuerdo qué axis colapsa cuál?**

**El axis que pasas es el que desaparece.** Si shape es `(3, 4)` y haces `sum(axis=0)`, queda `(4,)` — desapareció dim 0. Si `axis=1`, queda `(3,)`.

**❓ ¿`std` usa N o N-1?**

Default `ddof=0` (divide por N — desviación poblacional). Para desviación muestral (N-1), `arr.std(ddof=1)`. Pandas y scipy.stats usan N-1 por default — cuidado al comparar.

**❓ ¿`np.median` ignora NaN?**

No — usa `np.nanmedian`. Mismo patrón que `mean`/`nanmean`.

**❓ ¿Hay una agregación 'top-3' built-in?**

No directa. Usa `np.partition(arr, -3)[-3:]` para los 3 mayores (más rápido que sort completo). Si necesitas ordenados, `.sort()` después.

## 🔗 Referencias

- VanderPlas cap. 2 § 2.4
- [Statistics functions](https://numpy.org/doc/stable/reference/routines.statistics.html)

➡️ **Siguiente:** [017 — Broadcasting](../017-numpy-broadcasting/README.md)

## ✅ Soluciones de los ejercicios

Intentá resolverlos vos primero; acá está una solución de referencia comentada. Cada celda es autocontenida (re-importa lo que usa) y verifica el resultado con `assert`/`print`.

**Ejercicio 1.** Matriz 100x4 de ventas (filas=día, cols=tienda): calcula la media por tienda y por día.

In [ ]:
# Ejercicio 1 - Media por columna (tienda) y por fila (dia)
import numpy as np

rng = np.random.default_rng(0)
ventas = rng.integers(50, 500, size=(100, 4))   # 100 dias x 4 tiendas

media_tienda = ventas.mean(axis=0)   # colapsa filas -> un valor por tienda
media_dia = ventas.mean(axis=1)      # colapsa columnas -> un valor por dia
print("media por tienda:", media_tienda.round(1))
print("media por dia (primeros 3):", media_dia[:3].round(1))

# "el axis que pasas es el que desaparece":
assert media_tienda.shape == (4,)    # quedaron las 4 tiendas
assert media_dia.shape == (100,)     # quedaron los 100 dias
print("OK: axis=0 -> por tienda, axis=1 -> por dia")


**Ejercicio 2.** Para 1000 normales, reporta mean, std, median, p25, p75, min y max.

In [ ]:
# Ejercicio 2 - Estadisticos completos
import numpy as np

rng = np.random.default_rng(42)
x = rng.normal(0, 1, 1000)

stats = {
    "mean": x.mean(), "std": x.std(), "median": np.median(x),
    "p25": np.percentile(x, 25), "p75": np.percentile(x, 75),
    "min": x.min(), "max": x.max(),
}
for k, v in stats.items():
    print(f"  {k:6}: {v:+.3f}")

# Coherencia basica de los estadisticos:
assert stats["min"] <= stats["p25"] <= stats["median"] <= stats["p75"] <= stats["max"]
assert abs(stats["mean"]) < 0.2 and abs(stats["std"] - 1) < 0.1   # ~N(0,1)
print("OK: estadisticos ordenados y coherentes con N(0,1)")


**Ejercicio 3.** Inserta 50 NaN aleatorios en el array y compara `mean` (propaga) vs `nanmean`.

In [ ]:
# Ejercicio 3 - Missing data: mean vs nanmean
import numpy as np

rng = np.random.default_rng(1)
x = rng.normal(0, 1, 1000).copy()
idx = rng.choice(1000, size=50, replace=False)   # 50 posiciones al azar
x[idx] = np.nan

print("NaN presentes:", np.isnan(x).sum())
print("mean    (propaga):", x.mean())
print("nanmean (ignora) :", np.nanmean(x))

assert np.isnan(x.mean())            # un solo NaN ya contamina
assert not np.isnan(np.nanmean(x))   # nanmean usa solo los 950 validos
print("OK: nanmean ignora los NaN; mean los propaga")


**Ejercicio 4.** Genera retornos diarios aleatorios y calcula el precio acumulado con `cumprod(1 + r)`.

In [ ]:
# Ejercicio 4 - Precio acumulado con cumprod
import numpy as np

rng = np.random.default_rng(7)
retornos = rng.normal(0.0005, 0.01, 252)     # ~1 anio bursatil
precio = 100 * np.cumprod(1 + retornos)      # capitaliza desde 100

print("precio inicial:", round(precio[0], 2))
print("precio final  :", round(precio[-1], 2))

assert precio.shape == (252,)          # cumprod NO colapsa: mismo largo
assert (precio > 0).all()              # precios siempre positivos
assert np.isclose(precio[0], 100 * (1 + retornos[0]))
print("OK: cumprod(1+r) reconstruye la serie de precios")


**Ejercicio 5.** Con la matriz del ejercicio 1, usa `argmax(axis=0)` para encontrar el día de mayor venta de cada tienda.

In [ ]:
# Ejercicio 5 - argmax(axis=0): mejor dia por tienda
import numpy as np

rng = np.random.default_rng(0)
ventas = rng.integers(50, 500, size=(100, 4))   # misma matriz del ej. 1

mejor_dia = ventas.argmax(axis=0)     # indice (dia) del maximo de cada tienda
print("mejor dia por tienda:", mejor_dia)

# Verificamos que el indice apunta realmente al maximo de su columna:
for tienda in range(4):
    assert ventas[mejor_dia[tienda], tienda] == ventas[:, tienda].max()
assert mejor_dia.shape == (4,)        # un dia por cada tienda
print("OK: argmax(axis=0) da el dia pico de cada tienda")
